## 1.Minimal pipeline implementing agent with LangChain `create_agent` + Groq

In [ ]:
# %pip install -q langchain langchain-groq langchain-core langgraph

### 1.1 Access Model API key

### 1.2 Create react agent with llm, systemprompt, tools 

In [1]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import pipeline
from langchain_core.tools import tool


# --- Tools ---
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    return "It's 90 degrees and sunny."

@tool
def get_coolest_cities() -> str:
    """Get a list of the coolest cities."""
    return "nyc, sf"

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [get_weather, get_coolest_cities, multiply]

# --- HF Model ---
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=128,
    device_map="auto",
)

hf_pipeline = HuggingFacePipeline(pipeline=pipe)

llm3 = ChatHuggingFace(llm=hf_pipeline)

# --- System Prompt ---
systemPrompt = (
    "You are a helpful assistant. "
    "Use tools when needed to answer the user's question."
)

# --- Agent ---
agent = create_agent(
    model=llm3,  
    tools=tools,
    system_prompt=systemPrompt,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## 1.3.1 Run the Agent

In [2]:
# --- Run ---
user_question = "Will it rain tomorrow morning in Berlin?"

response = agent.invoke({
    "messages": [("user", user_question)]
})



[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [3]:
# raw response
print("raw response:")
print(response)

# Final answer
print("Final answer:")
print(response["messages"][-1].content)



print("reasoning steps:")
print(response["messages"][-1].additional_kwargs.get("reasoning_content"))


raw response:
{'messages': [HumanMessage(content='Will it rain tomorrow morning in Berlin?', additional_kwargs={}, response_metadata={}, id='a227ba21-0b04-4371-84ed-e950607478ef'), AIMessage(content="<|system|>\nYou are a helpful assistant. Use tools when needed to answer the user's question.</s>\n<|user|>\nWill it rain tomorrow morning in Berlin?</s>\n<|assistant|>\nYes, it is possible that tomorrow morning (Saturday) in Berlin could rain. The weather forecast provided by reputable websites such as the Weather Channel or AccuWeather usually includes information on potential weather conditions, including precipitation. If you are planning to travel to Berlin, you should confirm the weather conditions with local authorities or the hotel you are staying at.", additional_kwargs={}, response_metadata={}, id='lc_run--019ec789-8982-77d0-9ac8-65c03ce421ff-0', tool_calls=[], invalid_tool_calls=[])]}
Final answer:
<|system|>
You are a helpful assistant. Use tools when needed to answer the user'

In [4]:
# Optional: inspect full message trace (tool calls + results)
for msg in response["messages"]:
    print(f"[{msg.__class__.__name__}] {msg.content}")

[HumanMessage] Will it rain tomorrow morning in Berlin?
[AIMessage] <|system|>
You are a helpful assistant. Use tools when needed to answer the user's question.</s>
<|user|>
Will it rain tomorrow morning in Berlin?</s>
<|assistant|>
Yes, it is possible that tomorrow morning (Saturday) in Berlin could rain. The weather forecast provided by reputable websites such as the Weather Channel or AccuWeather usually includes information on potential weather conditions, including precipitation. If you are planning to travel to Berlin, you should confirm the weather conditions with local authorities or the hotel you are staying at.


### 1.3.2 Optinally: Streaming the response

## 2.Visualizing the llm's reasoning steps

In [5]:
from langchain.agents.middleware import AgentMiddleware


class TrajectoryLoggerMiddleware(AgentMiddleware):

    def before_model(self, state, runtime):
        print("\n================ MODEL =================")
        print("Messages sent to model:")
        print(state["messages"][-1])

    def after_model(self, state, runtime):
        print("\n================ MODEL RESPONSE =================")
        print(state["messages"][-1])

    def before_tool(self, tool_call, runtime):
        print("\n================ TOOL CALL =================")
        print(f"Tool: {tool_call['name']}")
        print(f"Args: {tool_call['args']}")

    def after_tool(self, tool_call, result, runtime):
        print("\n================ TOOL RESULT =================")
        print(result)


agent_debug = create_agent(
    model=llm3,
    tools=tools,
    system_prompt=systemPrompt,
    middleware=[
        TrajectoryLoggerMiddleware()
    ]
)

In [6]:
result = agent_debug.invoke(
    {
        "messages": [
            ("user", "What is 25 * 4?")
        ]
    }
)


================ MODEL =================
Messages sent to model:
content='What is 25 * 4?' additional_kwargs={} response_metadata={} id='182889c3-c280-4190-8d4e-48131918fba6'


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



================ MODEL RESPONSE =================
content="<|system|>\nYou are a helpful assistant. Use tools when needed to answer the user's question.</s>\n<|user|>\nWhat is 25 * 4?</s>\n<|assistant|>\nYes, I can help you with this question. The answer is 100." additional_kwargs={} response_metadata={} id='lc_run--019ec78a-d489-78b0-8e86-6ded542ec6c5-0' tool_calls=[] invalid_tool_calls=[]


## 3.Agent Evaluation

In [ ]:
# !pip install agentevals

In [ ]:
def print_trajectory(messages):
    print("\n===== TRAJECTORY =====")

    for i, msg in enumerate(messages):
        print(f"\n[{i}] {msg.__class__.__name__}")
        print(msg.content)

        if hasattr(msg, "tool_calls"):
            print("tool_calls =", msg.tool_calls)

## 3.1 Trajectory Match Evaluator

In [ ]:
from agentevals.trajectory.match import (
    create_trajectory_match_evaluator
)

trajectory_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="strict"
)

outputs = agent.invoke(
    {
        "messages": [
            ("user", "What is 3 * 5?")
        ]
    }
)

print_trajectory(outputs["messages"])

reference_outputs = {
    "messages": outputs["messages"]
}

In [ ]:
score = trajectory_evaluator(
    outputs=outputs["messages"],
    reference_outputs=reference_outputs["messages"]
)

print(score)

## 3.2 LLM-as-Judge

In [ ]:
from agentevals.trajectory.llm import (
    create_trajectory_llm_as_judge,
    TRAJECTORY_ACCURACY_PROMPT
)

trajectory_judge = create_trajectory_llm_as_judge(
    model="groq:qwen/qwen3-32b"  # <-- Use the "groq:" prefix string here
)

In [ ]:
outputs = agent.invoke(
    {
        "messages": [
            ("user", "What is the weather in SF?")
        ]
    }
)

print_trajectory(outputs["messages"])

evaluation = trajectory_judge(
    outputs=outputs["messages"]
)

print("\n===== JUDGE RESULT =====")
print(evaluation)

## 4.some interesting middlewares

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.agents import create_agent         

# 1. Define the policy
hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "multiply": True,
        "get_weather": True,
    }
)

# 2. Compile the correct agent with the checkpointer attached
agent_hitl = create_agent(
    model=llm,
    tools=tools,
    checkpointer=InMemorySaver(), 
    middleware=[hitl]
)

# 3. Setup configurations
config = {"configurable": {"thread_id": "real-interactive-demo-session"}}
initial_input = {"messages": [("user", "What is the current weather in San Francisco right now?")]}

print("🚀 Starting Agent Execution...")
response = agent_hitl.invoke(initial_input, config=config)

# --- 4. Read the True Graph State to check for an active Interruption ---
current_graph_state = agent_hitl.get_state(config)

if current_graph_state.next:
    print("\n🛑 [HITL INTERRUPT TRIGGERED] The agent is paused awaiting human review!")
    
    active_tasks = current_graph_state.tasks
    if active_tasks and active_tasks[0].interrupts:
        interrupt_info = active_tasks[0].interrupts[0]
        
        print(f"\n👉 Pending Tool Target: {interrupt_info.value['action_requests'][0]['name']}")
        print(f"👉 Target Arguments   : {interrupt_info.value['action_requests'][0]['args']}")
        
        # -------------------------------------------------------------
        # 🛑 LIVE INTERACTION: The notebook will stop and wait for you!
        # -------------------------------------------------------------
        user_choice = input("\n👥 Type 'approve' to execute the tool, or 'reject' to stop it: ").strip().lower()
        
        if user_choice not in ["approve", "reject"]:
            print("⚠️ Invalid choice. Defaulting to reject for safety.")
            user_choice = "reject"
            
        # Dynamically map your keyboard input into the Command decision engine
        resume_command = Command(resume={"decisions": [{"type": user_choice}]})
        
        print(f"\n🔄 Resuming execution with decision: {user_choice.upper()}...")
        response = agent_hitl.invoke(resume_command, config=config)

# --- 5. Print final response ---
print("\n✨ Final Agent Answer:")
print(response["messages"][-1].content)